## Prepare questions from NIFD for training GRPO

Created by: Sahana Kowshik

Date: 05/07/2025

In [1]:
import pandas as pd
import json
import random

In [2]:
directory_path = "/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nifd"
train = pd.read_csv(f"{directory_path}/train.csv")

In [3]:
len(train)

292

In [4]:
train['DX'].value_counts()

DX
CON     136
BV       77
PNFA     40
SV       39
Name: count, dtype: int64

In [5]:
# # Function to assign ground truth label and MCI question
# def add_etpr(row):
#     if row['DX'] == 'CON':
#         row['ETPR'] = "NC"
#     elif row['DX'] in ['BV', 'PNFA', 'SV']:
#         row['ETPR'] = "FTLD"

#     return row

In [6]:
columns = ['ID', 'patient_summary', 'diag_summary', 'question', 'options', 'ground_truth', 'ground_truth_text', 'COHORT']

In [7]:
import random, string

etiology_question_variants = [
    "Identify the patient's subtype of Frontotemporal Dementia using the information provided, if applicable.",
    "Based on the information provided, determine the subtype of Frontotemporal Dementia, if applicable.",
    "If applicable, use the provided information to identify the subtype of Frontotemporal Dementia.",
    "Determine the Frontotemporal Dementia subtype for the patient using the available information, if applicable.",
    "Using the information provided, classify the patient's Frontotemporal Dementia subtype, if applicable."
]


# Raw etiology answer texts in original order
ETIOLOGY_OPTION_TEXTS = [
    "Behavioral variant frontotemporal dementia",
    "Progressive non-fluent aphasia",
    "Semantic variant primary progressive aphasia",
    "Not applicable (no cognitive impairment)",
]

# Mapping of ETPR code to correct answer text
ETPR_ANSWER_MAP = {
    'BV': ETIOLOGY_OPTION_TEXTS[0],
    'PNFA': ETIOLOGY_OPTION_TEXTS[1],
    'SV': ETIOLOGY_OPTION_TEXTS[2],
    'CON': ETIOLOGY_OPTION_TEXTS[3],
}

def add_etpr_question(row, *, seed=None):
    rng = random.Random(seed if seed is not None else row.name)

    # 1️⃣ Randomly select a question variant
    row['question'] = rng.choice(etiology_question_variants)

    # 2️⃣ Shuffle options and assign fresh letters
    shuffled = ETIOLOGY_OPTION_TEXTS.copy()
    rng.shuffle(shuffled)
    letters = list(string.ascii_uppercase[:len(shuffled)])
    row['options'] = "\n".join(f"{ltr}. {opt}" for ltr, opt in zip(letters, shuffled))

    # 3️⃣ Get correct text from ETPR code (default to "Not applicable" if unknown)
    correct_text = ETPR_ANSWER_MAP.get(row['DX'])

    if correct_text is None:
        raise ValueError(f"Invalid ETPR value: {row['DX']}")


    # 4️⃣ Get ground truth letter from shuffled list
    if correct_text in shuffled:
        row['ground_truth'] = letters[shuffled.index(correct_text)]
    else:
        row['ground_truth'] = None  # shouldn't happen unless text mismatch
        
    row['ground_truth_text'] = correct_text

    return row


In [8]:
# Apply the function to the MCI training set
train = train.apply(add_etpr_question, axis=1)

In [9]:
train[train['ground_truth'].isna()]

,LONI_ID,CLINICAL_LINKDATE,DX,SITE,VISIT_NUMBER,DOB,GENDER,EDUCATION,RACE,CDR_DCDATE,...,MDEXAM_UHDRS,MDEXAM_UPDRS,COHORT,brain_findings_summary,patient_summary,diag_summary,question,options,ground_truth,ground_truth_text


In [10]:
train.iloc[0]['patient_summary']

'{"Subject Demographics": {"Gender": "Male", "Years of Education": 15, "Race": "White"}, "Neuropsychological Battery Summary Scores": {"Montreal Cognitive Assessment": 14, "Modified Benson Figure": 8}}'

In [11]:
train["ID"] = train["LONI_ID"]
train = train[columns]
print(len(train))
train = train.dropna(how='any').reset_index(drop=True)
print(len(train))

292
292


/scratch/9210280.1.cbm.q/ipykernel_2374428/3149451496.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train["ID"] = train["LONI_ID"]


In [12]:
print(train.iloc[-1]['diag_summary'])

{"Diagnosis": "PNFA"}


In [13]:
train['ground_truth_text'].value_counts()

ground_truth_text
Not applicable (no cognitive impairment)        136
Behavioral variant frontotemporal dementia       77
Progressive non-fluent aphasia                   40
Semantic variant primary progressive aphasia     39
Name: count, dtype: int64

In [14]:
train['Q_TYPE'] = "FTD subtype"

In [15]:
train.head()

,ID,patient_summary,diag_summary,question,options,ground_truth,ground_truth_text,COHORT,Q_TYPE
0,1_S_0001,"{""Subject Demographics"": {""Gender"": ""Male"", ""Y...","{""Diagnosis"": ""CON""}",Determine the Frontotemporal Dementia subtype ...,A. Semantic variant primary progressive aphasi...,D,Not applicable (no cognitive impairment),NIFD,FTD subtype
1,1_S_0002,"{""Subject Demographics"": {""Gender"": ""Female"", ...","{""Diagnosis"": ""BV""}","Based on the information provided, determine t...",A. Semantic variant primary progressive aphasi...,D,Behavioral variant frontotemporal dementia,NIFD,FTD subtype
2,1_S_0003,"{""Subject Demographics"": {""Gender"": ""Male"", ""Y...","{""Diagnosis"": ""SV""}",Identify the patient's subtype of Frontotempor...,A. Semantic variant primary progressive aphasi...,A,Semantic variant primary progressive aphasia,NIFD,FTD subtype
3,1_S_0005,"{""Subject Demographics"": {""Gender"": ""Male"", ""Y...","{""Diagnosis"": ""BV""}","Based on the information provided, determine t...",A. Behavioral variant frontotemporal dementia\...,A,Behavioral variant frontotemporal dementia,NIFD,FTD subtype
4,1_S_0006,"{""Subject Demographics"": {""Gender"": ""Male"", ""Y...","{""Diagnosis"": ""BV""}","Based on the information provided, determine t...",A. Not applicable (no cognitive impairment)\nB...,C,Behavioral variant frontotemporal dementia,NIFD,FTD subtype


In [16]:
train.to_csv(f"{directory_path}/training_data/training_data_grpo/train_with_questions.csv", index=False)

In [17]:
question_df = train.groupby(['question', 'Q_TYPE']).size().reset_index(name='count').sort_values(by=['Q_TYPE', 'count'])

In [18]:
# Save as CSV
csv_path = f"{directory_path}/training_data/training_data_grpo/train_question_counts.csv"
question_df.to_csv(csv_path, index=False)